## Automatización diaria (ejecución automática a las 06:00)

> *Colab no puede “despertarse solo” cuando nadie lo abre, así que la forma práctica de automatizar es descargar el script y programarlo en una máquina o servicio que esté siempre encendido.*

### Paso 1 – Exportar el código
- En Colab: **Archivo → Descargar → Python (.py)**  
- Esto genera un archivo `cumples.py` con todo el flujo listo para ejecutar.

### Paso 2 – Programar la ejecución

**En Linux / Mac (cron):**

0 6 * * * /usr/bin/python3 /ruta/a/cumples.py

Este cron job corre el script todos los días a las 06:00 de la zona horaria del sistema.

In [ ]:
import pandas as pd
!pip install pandasql
import pandasql as psql

# Leemos la pestaña del Google Sheet como si fuera una tabla SQL
sheet_id = "1T5Tk4RrvlkawgJ1WNRJVDNtI9j2ykAWvY1bZpKeHoXA"
gid = "419565292"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df = pd.read_csv(url)

# --- Simulación de consulta SQL ---
query = """
/*  En una base real filtraríamos por la comunidad:
    WHERE instanceId = 9366
    pero esta hoja no tiene esa columna,
    así que lo dejamos comentado como referencia. */
SELECT
  "id usuario"   AS employee_id,
  "Usuario"      AS employee_email,
  "name"         AS employee_name,
  "birthdayDate" AS birthdayDate,
  "ChiefId"      AS chief_id,
  "Chief"        AS chief_label,   -- si lo necesitás
  "Chief name"   AS chief_name,
  "Chief email"  AS chief_email
FROM df
"""
resultado = psql.sqldf(query, locals())
resultado.head()


  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=ad6f151310b1e9294d4c0c3d1789d81a5b073a6fa29b8816074d00d7a4bb755e
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


,employee_id,employee_email,employee_name,birthdayDate,chief_id,chief_label,chief_name,chief_email
0,1197571,eze@humand.co,Ezequiel Morkin,06/23/2024,1197757,nathalie.rovayo@empresatres.com,Nathalie Rovayo,nathalie.rovayo@empresatres.com
1,1197577,ivan@humand.co,Iván Ruschanoff,04/21/2024,1197757,nathalie.rovayo@empresatres.com,Nathalie Rovayo,nathalie.rovayo@empresatres.com
2,1197615,santos@humand.co,Santos Bengolea,12/01/2024,1197757,nathalie.rovayo@empresatres.com,Nathalie Rovayo,nathalie.rovayo@empresatres.com
3,1197689,rommel.gallegos@empresatres.com,Rommel Gallegos,08/30/2024,1197757,nathalie.rovayo@empresatres.com,Nathalie Rovayo,nathalie.rovayo@empresatres.com
4,1197749,alejandro.martin.hernandez@empresatres.com,Alejandro Hernandez,21/02/2024,1197757,nathalie.rovayo@empresatres.com,Nathalie Rovayo,nathalie.rovayo@empresatres.com


## Procesamiento

- La columna `birthdayDate` se convierte a tipo fecha (`datetime`) para poder comparar.  
- Se toma la fecha actual en la zona horaria del cliente (`America/Argentina/Buenos_Aires`).  
- Se filtran los registros donde mes y día coincidan con la fecha actual, quedando solo los empleados que cumplen años hoy.


In [ ]:
from datetime import datetime
import pytz  # si querés manejar zona horaria

# 1) Convertir birthdayDate a datetime
resultado['birthdayDate'] = pd.to_datetime(
    resultado['birthdayDate'],
    errors='coerce',        # pone NaT si hay valores raros
    dayfirst=False,         # la hoja venía en mm/dd/yyyy
)

# 2) Fecha de hoy con la zona horaria del cliente
#    (cambiá 'America/Argentina/Buenos_Aires' por la que necesites)
tz = pytz.timezone('America/Argentina/Buenos_Aires')
hoy = datetime.now(tz)
hoy_mmdd = hoy.strftime('%m-%d')

# 3) Filtrar los cumpleañeros de hoy
cumple_hoy = resultado[
    resultado['birthdayDate'].dt.strftime('%m-%d') == hoy_mmdd
]

cumple_hoy

,employee_id,employee_email,employee_name,birthdayDate,chief_id,chief_label,chief_name,chief_email


In [ ]:
# === CELDA DE PRUEBA ===
# Reemplaza temporalmente el DataFrame real por uno de prueba
from datetime import datetime
import pandas as pd

cumple_hoy = pd.DataFrame([{
    'employee_name': 'Empleado Demo',
    'chief_name':   'Jefe Demo',
    'chief_email':  'tu_email_de_prueba@loquesea.com',
    'birthdayDate': datetime.now()
}])


In [ ]:
# === ENVÍO DE CORREOS (smtplib) ===
import smtplib
from email.message import EmailMessage

# --- CONFIGURACIÓN SMTP (modificá estos 4 valores) ---
SMTP_HOST = "smtp.gmail.com"      # Gmail: smtp.gmail.com | Outlook: smtp.office365.com
SMTP_PORT = 587
SMTP_USER = "TU_CUENTA@gmail.com" # remitente
SMTP_PASS = "TU_PASS_APP"         # contraseña de aplicación (no tu password normal)

FROM_NAME = "Notificaciones RH"
REPLY_TO  = "TU_CUENTA@gmail.com"

# --- MODO PRUEBA ---
DRY_RUN = True  # True = NO envía, solo muestra. Cambiar a False para enviar de verdad.

def build_email(to_addr, chief_name, employee_name, employee_bday):
    """Crea el email (texto + HTML simple)."""
    subject = f"Recordatorio: hoy cumple {employee_name}"
    text = f"""Hola {chief_name},

Hoy {employee_name} cumple años ({employee_bday.strftime('%d/%m')}).
Sugerencia: envíale un saludo o compártelo en el canal del equipo.

— Este mensaje es automático.
"""
    html = f"""\
    <html>
      <body>
        <p>Hola <b>{chief_name}</b>,</p>
        <p>Hoy <b>{employee_name}</b> cumple años (<b>{employee_bday.strftime('%d/%m')}</b>).</p>
        <p>Sugerencia: envíale un saludo o compártelo en el canal del equipo.</p>
        <br>
        <p style="color:#777">— Este mensaje es automático.</p>
      </body>
    </html>
    """

    msg = EmailMessage()
    msg["Subject"] = subject
    msg["From"]    = f"{FROM_NAME} <{SMTP_USER}>"
    msg["To"]      = to_addr
    msg["Reply-To"]= REPLY_TO
    msg.set_content(text)
    msg.add_alternative(html, subtype="html")
    return msg

def send_one(msg):
    """Envía un único correo. Con DRY_RUN imprime y no envía."""
    if DRY_RUN:
        print(f"[PRUEBA] A: {msg['To']} | Asunto: {msg['Subject']}")
        return True

    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.starttls()
            server.login(SMTP_USER, SMTP_PASS)
            server.send_message(msg)
        print(f"[OK] Enviado a {msg['To']}")
        return True
    except Exception as e:
        print(f"[ERROR] {msg['To']}: {e}")
        return False

# === PIPELINE FINAL: si hay cumpleañeros, enviar; si no, terminar en silencio limpio ===
if cumple_hoy.empty:
    print("No hay cumpleaños hoy. No se envía ningún correo.")
else:
    enviados, errores = 0, 0
    for r in cumple_hoy.itertuples(index=False):
        # Columnas esperadas del paso anterior:
        # employee_name, chief_name, chief_email, birthdayDate
        if not isinstance(r.chief_email, str) or "@" not in r.chief_email:
            print(f"[SKIP] Email inválido para jefe de {r.employee_name}: {r.chief_email}")
            errores += 1
            continue

        msg = build_email(
            to_addr      = r.chief_email.strip(),
            chief_name   = r.chief_name,
            employee_name= r.employee_name,
            employee_bday= r.birthdayDate
        )
        ok = send_one(msg)
        enviados += int(ok)
        errores  += int(not ok)

    print(f"\nResumen: enviados={enviados} | errores={errores} | modo_prueba={DRY_RUN}")


[PRUEBA] A: tu_email_de_prueba@loquesea.com | Asunto: Recordatorio: hoy cumple Empleado Demo

Resumen: enviados=1 | errores=0 | modo_prueba=True
